# Imports

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import os
from sklearn.metrics import f1_score, accuracy_score, mean_squared_error, mean_absolute_error, r2_score, classification_report, roc_auc_score
import plotly.graph_objects as go
import plotly.figure_factory as ff
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from tqdm import tqdm

# Data loading

In [2]:
dataset_name = "larfield_activity"

TASK_LIST = ["binary"]

task = "binary"
dataset_name_title = "larfield_activity"

classification_head = "ClassificationHeadMLP"

classess_labels = ["stationary", "non-stationary"]

num_classes = len(classess_labels)
print("dataset_name, task:", dataset_name, task)
print("classess_labels:", classess_labels)

dataset_name, task: larfield_activity binary
classess_labels: ['stationary', 'non-stationary']


In [3]:
results = {}
# TODO: set experiment path
EXP_PATH = f"/home/emognition/Desktop/RL-PPG/downstream_tasks/experiments/{dataset_name_title}"
logs_paths = [os.path.join(f"{EXP_PATH}/logs_test_pred/", lp) for lp in os.listdir(f"{EXP_PATH}/logs_test_pred/")]

for logs_path in logs_paths:
    #print(logs_path)
    if "all_subjects" in logs_path:
        continue
    if task in logs_path:
        results[logs_path.split("/")[-1].replace("pretrained", "initialized")] = {}
        for fold in os.listdir(logs_path):
            if classification_head in fold:
                results[logs_path.split("/")[-1].replace("pretrained", "initialized")][fold] = {}
                for result_file in os.listdir(os.path.join(logs_path, fold)):
                    try:
                        if "predicted_labels" in result_file:
                            results[logs_path.split("/")[-1].replace("pretrained", "initialized")][fold]["predicted_labels"] = np.load(os.path.join(logs_path, fold, result_file)).squeeze()
                        elif "true_labels" in result_file:
                            results[logs_path.split("/")[-1].replace("pretrained", "initialized")][fold]["true_labels"] = np.load(os.path.join(logs_path, fold, result_file)).squeeze()
                    except Exception as e:
                        results[logs_path.split("/")[-1].replace("pretrained", "initialized")][fold]["predicted_labels"] = None
                        results[logs_path.split("/")[-1].replace("pretrained", "initialized")][fold]["true_labels"] = None
                        print("ERROR:",e)
            
old_keys = list(results.keys())[:]
for key in old_keys:
    if "ACC_ECG" in key:        
        results[key.replace("ACC_ECG", "ECG_ACC")] = results.pop(key)
        print(key)
        
len(list(results.keys()))

16

# Count metrics

In [4]:
for experiment, all_folds in results.items():
    for fold, results_labels in all_folds.items():
        try:
        # TODO: czy to dobrze że co iterację potencjalnie może być brana inna wartość dla pos_label dla zadania gender ??? czy to nie powinno być hardcoded
            all_folds[fold]["f1_score"] = f1_score(results_labels["true_labels"], results_labels["predicted_labels"],  average= 'macro')      
            all_folds[fold]["accuracy"] = accuracy_score(results_labels["true_labels"], results_labels["predicted_labels"])        
        except Exception as e:
            all_folds[fold]["f1_score"] = None
            all_folds[fold]["accuracy"] = None
            print("ERROR:",experiment,e)


# Count baselines

In [5]:
all_true_labels = {}
task_baselines = {}
for experiment, all_folds in results.items():
    if "across_time" in experiment and "cold_start" in experiment:
        for fold, results_labels in all_folds.items():
            all_true_labels[fold] = results_labels["true_labels"].tolist()
        break # only one experiment is needed; no matter which one
    else:
        for fold, results_labels in all_folds.items():
            all_true_labels[fold] = results_labels["true_labels"].tolist()
        break
         # only one experiment is needed; no matter which one
for dummy_strategy in ["most_frequent", "prior", "stratified", "uniform"]: # , "constant"
    task_baselines[dummy_strategy] = {}
    for fold, true_labels in all_true_labels.items():
        clf = DummyClassifier(strategy=dummy_strategy, random_state=321)
        clf.fit(list(range(len(true_labels))), true_labels)
        dummy_preds = clf.predict(list(range(len(true_labels))))     
        dummy_f1_score =  f1_score(y_true=true_labels, y_pred=dummy_preds,average="macro")
        task_baselines[dummy_strategy][fold] = dummy_f1_score

mean_task_baselines = {}
for key, value in task_baselines.items():
    mean_task_baselines[key] = {}
    fscores = [fscore for fold, fscore in value.items()]
    mean_task_baselines[key]["f1_score_mean"] = np.mean(fscores)
    mean_task_baselines[key]["f1_score_std"] = np.std(fscores)
    
        
mean_task_baselines
all_labels = []
for key, value in all_true_labels.items():
    all_labels += value
all_labels.count(0)
mean_task_baselines
task_baselines_df = pd.DataFrame(task_baselines).reset_index()
print("Most frequent:", np.round(task_baselines_df["most_frequent"].mean(), 3),  "\pm", np.round(task_baselines_df["most_frequent"].std(), 3))
print("Uniform:", np.round(task_baselines_df["uniform"].mean(), 3),  "\pm", np.round(task_baselines_df["uniform"].std(), 3))

Most frequent: 0.411 \pm 0.029
Uniform: 0.488 \pm 0.064


# Visualize metrics

In [6]:
f1_scores, accuracies, aucs, experiment_descriptions = [], [], [], []
backbones, folds, modalities, tasks = [], [], [], []
for experiment, all_folds in results.items():
    for fold, results_labels in all_folds.items():
        f1_scores.append(all_folds[fold]["f1_score"])
        accuracies.append(all_folds[fold]["accuracy"])
        experiment_descriptions.append(experiment)
        folds.append(fold)

all_results = pd.DataFrame()
all_results["Accuracy"] = accuracies
all_results["F1_score"] = f1_scores
all_results["fold"] = folds
all_results["experiment_description"] = experiment_descriptions

modalities = []
for ed in experiment_descriptions:
    try:
        modalities.append(ed.split("__")[-1].split("_test")[0])
    except:
        modalities.append("ECG")
all_results["modality"] = modalities
all_results["lr"] = ["0."+e.split(".")[1].split("_")[0] for e in all_results["experiment_description"].values]

all_results["finetuning"] = [True if "finetune" in r["experiment_description"] else False for i,r in all_results.iterrows()]
all_results["across_time"] = [True if "across_time" in r["experiment_description"] else False for i,r in all_results.iterrows()]
all_results["cold_start"] = [True if "cold_start" in r["experiment_description"] else False for i,r in all_results.iterrows()]
all_results["personal"] = [True if "personal" in r["experiment_description"] else False for i,r in all_results.iterrows()]
all_results["from_scratch"] = [True if "not_loaded_checkpoint" in r["experiment_description"] else False for i,r in all_results.iterrows()]
all_results["personal_ssl"] = [True if "personal" in r["experiment_description"].split("__")[0] else False for i,r in all_results.iterrows()]

## Aggregating results

In [7]:
aggregated_results_mean = all_results.groupby(by=["experiment_description","finetuning", "across_time","cold_start","personal","from_scratch", "personal_ssl"])[["F1_score", "Accuracy"]].mean().add_suffix("_mean").reset_index()#.agg(['mean', 'std']).add_prefix()
aggregated_results_std = all_results.groupby(by=["experiment_description","finetuning","across_time","cold_start","personal","from_scratch", "personal_ssl"])[["F1_score", "Accuracy"]].std().add_suffix("_std").reset_index()#.agg(['mean', 'std']).add_prefix()
aggregated_results_mean["lr"] = ["0."+e.split(".")[1].split("_")[0] for e in aggregated_results_mean["experiment_description"].values]
aggregated_results_mean["train_size"] = [int(e.split("train_size_ablation_")[1].split("__")[0]) for e in aggregated_results_mean["experiment_description"].values]

In [8]:
aggregated_results = pd.merge(aggregated_results_mean, aggregated_results_std, on=["experiment_description"])
modalities = []
experiment_description_short = []

for ed in aggregated_results["experiment_description"]:
    if len(ed.split("__")) == 3:
        model_name, description, modality = ed.split("__")
    else:
        model_name_description, modality = ed.split("__")
        
        model_name_description = model_name_description.split("_")
        model_name, description = '_'.join(model_name_description[:-4]), '_'.join(model_name_description[-4:])
        model_name_description = None

    model_name = model_name.replace("unimodal_fusion", 'unimodal')

    for to_remove in ['initial_', 'ts_tcc_Larfield_', '_ecg', '_acc']:
        model_name = model_name.replace(to_remove, '')

    if description.split('_')[-1].endswith('classes'):
        description = '_'.join(description.split('_')[:-1])

    for to_remove in ['binary', 'multiclass', 'fusion_mean_', task]:
        description = description.replace(to_remove, '')

    description = description.replace('full_representation', 'FR')
    description = description.replace('feature_learning', 'FL')

    model_name = model_name.replace('full_representation', 'FR')
    model_name = model_name.replace('feature_learning', 'FL')
    
    if len(description) > 0 and description[0] != '_':
        description = '_' + description

    modalities.append(modality.replace('_test', ''))
    if "unimodal_fusion" in ed and "not_loaded_checkpoint" not in ed:
        experiment_description_short.append(f"unimodal")
    elif "feature_learning" in ed or "FL" in model_name:
        experiment_description_short.append(f"{model_name}{description}".split("_fix")[0].split("_2classes")[0]+"_FL")
    elif "not_loaded_checkpoint" in ed:
        experiment_description_short.append(f"multimodal_not_loaded_checkpoint")

    else:
        experiment_description_short.append(f"{model_name}")

aggregated_results["modality"] = modalities
aggregated_results["experiment_description_short"] = experiment_description_short


In [9]:
#SELECTED BY THIS
groups_for_selecting = aggregated_results.groupby(by=["across_time_x","cold_start_x", "personal_x", "finetuning_x", "lr"])["F1_score_mean"].mean().reset_index().groupby(by=["across_time_x","cold_start_x", "personal_x"])
best_df = pd.DataFrame()

    
for i, g in groups_for_selecting:
    g_max = g.sort_values(by="F1_score_mean", ascending=False).reset_index(drop=True)[:1]
    wanted_value = pd.DataFrame()
    for c in [ "across_time_x","cold_start_x", "personal_x", "lr"]:
        if len(wanted_value) == 0:
            wanted_value = aggregated_results[aggregated_results[c] == g_max[c].values[0]]
        else: 
            wanted_value = wanted_value[wanted_value[c] == g_max[c].values[0]]
    
    if len(best_df) == 0:
        best_df = wanted_value
    else:
        best_df = pd.concat([best_df, wanted_value])
best_df

best_df_latex = pd.DataFrame()
method = []
loso = []

for index, row in best_df.iterrows():
    if row["finetuning_x"] == True:
        if "not_loaded_checkpoint" in row["experiment_description_short"]:
            method.append("finetuned_fs_encoder")
        else:
            method.append("finetuned_rl_ppg_encoder")
    else:
        if "not_loaded_checkpoint" in row["experiment_description_short"]:
            method.append("frozen_fs_encoder")
        else:
            method.append("frozen_rl_ppg_encoder")
    if row[["across_time_x","cold_start_x", "personal_x"]].tolist() == [False, False, False]:
        loso.append(f"{np.round(row['F1_score_mean'], 3)} \pm {np.round(row['F1_score_std'], 3)}")


    
best_df_latex["Method"] = method
best_df_latex["LOSO"] = loso
best_df_latex

,Method,LOSO
0,frozen_rl_ppg_encoder,0.879 \pm 0.05
1,finetuned_rl_ppg_encoder,0.873 \pm 0.045
2,frozen_fs_encoder,0.805 \pm 0.072
3,finetuned_fs_encoder,0.851 \pm 0.055
